In [ ]:
%matplotlib inline
from shared_setup import *
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.lines import Line2D
from collections import OrderedDict

experiment, info = load_data()
print(f"Mode: {info['mode']}")

In [ ]:
animal_ID = 'SS16'
animal = experiment.get_animal(animal_ID)

In [ ]:
phase = 'uniform_training_last5'
sessions = select_sessions(animal, preset = phase)
clean_trials = filter_trials(sessions, min_trials = 10, exclude_abort = True, exclude_opto = False)

In [ ]:
um_results_pooled = compute_um(clean_trials, 
                        mode = 'pooled' # per_session or pooled
                        )
psyc_pooled = compute_psychometric(clean_trials,
						mode = 'pooled' # per_session or pooled
						)
um_results_session = compute_um(clean_trials, 
                        mode = 'per_session' # per_session or pooled
                        )
psyc_session = compute_psychometric(clean_trials,
						mode = 'per_session' # per_session or pooled
						)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
plot_um(um_results_pooled, ax = axes[0])
axes[0].title.set_text(f"Update Matrix")
plot_psychometric(psyc_pooled, ax = axes[1])
axes[1].title.set_text(f"Psychometric Curve")
plt.suptitle(f"UM - Pooled - {phase}")
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(2, 5, figsize=(15, 5))
for i in range(axes.shape[1]):
    session_idx = um_results_session['per_session'][i]['session_idx']
    plot_um(um_results_session, ax = axes[0,i], session_idx = session_idx)
    axes[0,i].set_title(f"Session {session_idx}")
    plot_psychometric(psyc_session['per_session'][i], ax = axes[1,i])
fig.suptitle(f"UM - Per Session - {phase}")
plt.tight_layout()
plt.show()

In [ ]:
phase = 'uniform_opto'
uniform_opto_sessions = select_sessions(animal, preset = phase)
opto_trials = filter_trials(uniform_opto_sessions, mask_fn = lambda s: opto_mask(s.trials, delta= 0))
post_opto_trials = filter_trials(uniform_opto_sessions, mask_fn = lambda s: opto_mask(s.trials, delta= 1))
non_opto_trials = filter_trials(uniform_opto_sessions, exclude_opto = True)

In [ ]:
opto_um = compute_um(opto_trials, mode = 'pooled')
post_opto_um = compute_um(post_opto_trials, mode = 'pooled')
non_opto_um = compute_um(non_opto_trials, mode = 'pooled')
opto_psyc = compute_psychometric(opto_trials, mode = 'pooled')
post_opto_psyc = compute_psychometric(post_opto_trials, mode = 'pooled')
non_opto_psyc = compute_psychometric(non_opto_trials, mode = 'pooled')

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for i, (um, psyc, title) in enumerate(zip([opto_um, post_opto_um, non_opto_um],
										   [opto_psyc, post_opto_psyc, non_opto_psyc],
										   ['Opto Trials', 'Post-Opto Trials', 'Non-Opto Trials'])):
	plot_um(um, ax = axes[0,i])
	axes[0,i].set_title(f"{title} - UM")
	plot_psychometric(psyc, ax = axes[1,i])
	axes[1,i].set_title(f"{title} - Psychometric")
plt.suptitle(f"UM and Psychometric - {phase}", fontsize=16)
plt.tight_layout()
plt.show()